# AIM-Flow: official T2I-CompBench + GenEval evaluation

This is the complete, executable official evaluation for three guidance methods over one locked Stable Diffusion 3 Medium checkpoint:

1. **Our Method** — the repository's Sparse Primitive Flow Composition (SPFC).
2. **Base CFG** — the repository's standard SD3 classifier-free-guidance path.
3. **Rectified CFG++** — the authors' unmodified, commit-pinned custom SD3 pipeline from the official repository.

Part 1 uses the official T2I-CompBench BLIP-VQA and UniDet scripts for Color, Texture, Spatial, and Shape. Part 2 uses the official 553-prompt, four-images-per-prompt GenEval metadata, Mask2Former evaluator, and `summary_scores.py`. No surrogate or auxiliary metrics are computed.

**Apples-to-apples protocol.** T2I-CompBench uses 300 prompts × 10 images in each selected category. GenEval uses 553 prompts × 4 images. Every method uses the same immutable SD3 Medium revision, checkpoint scheduler, 1024×1024 resolution, 28 inference steps, guidance 4.5, empty negative prompt, float16 precision, prompt order, and per-image seeds. These shared settings follow the official Rectified-CFG++ paper/repository wherever applicable. Rectified CFG++'s `sigma_noise=0.005` is intrinsic to that method. All methods generate one image per call, matching the official demo and avoiding any batch-dependent RNG or memory difference.

**Documented model adaptation.** The Rectified-CFG++ paper evaluates both SD3 Medium and SD3.5 Large at 1024×1024; the repository demo happens to instantiate SD3.5 Large. This notebook instead applies the repository's official custom pipeline to its supported SD3 Medium backbone because Our Method is implemented for that checkpoint and the requested comparison requires one identical model. No Rectified-CFG++ equations or pipeline code are rewritten. The SPFC aggregation schedule is the repository's 24-step schedule mapped monotonically to the same relative locations in the shared 28-step trajectory; this method-internal schedule is recorded in the manifest.

**Environment deviation.** The paper reports an A100 40 GB, Python 3.10, and PyTorch 2.0.1. Generation here uses pinned current packages and CPU offload so the common SD3 checkpoint runs on this PC's Turing GPUs; the official T2I-CompBench and GenEval scorers remain in their own pinned legacy environments. This changes hardware/runtime, not prompts, weights, scheduler, sampling settings, or guidance equations. The Rectified-CFG++ repository has no installable package metadata at the pinned commit, so it is installed operationally by cloning the exact source and loading its custom pipeline by path after installing pinned dependencies.

Protocol sources: [official Rectified-CFG++ repository](https://github.com/shreshthsaini/Rectified-CFGpp/tree/master), [project page](https://rectified-cfgpp.github.io/), and [paper](https://arxiv.org/abs/2510.07631). The exact repository commit and SHA-256 hashes of `demo.py` and the custom pipeline are asserted at runtime.

The official GenEval sample script resets one global RNG stream per prompt and leaves model-specific inference defaults configurable. Here, seeds 42–45 are assigned per image and the SD3 settings are explicit so initial noise is matched across all three implementations. Official prompt metadata/order, four-image count, directory layout, Mask2Former evaluation, and summary script are preserved. The COCO-named Mask2Former checkpoint below is GenEval's own required detector, not COCO prompts, images, or FID evaluation.


In [1]:
# Installation and repository pins. Run this cell before importing torch/diffusers.
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "aim_flow").is_dir():
            return candidate.resolve()
    raise FileNotFoundError("Launch this notebook from inside the aim-flow repository.")


REPO_ROOT = find_repo_root(Path.cwd())
os.chdir(REPO_ROOT)

RUN_INSTALLS = True
PINS = {
    "rectified_cfgpp": ("https://github.com/shreshthsaini/Rectified-CFGpp.git", "3c838882f7b2cdc2e1c3e5785468cdcb1fb27190"),
    "t2i_compbench": ("https://github.com/Karine-Huang/T2I-CompBench.git", "1b7094991a57f3c22abdd4f6e8ba6c1a15517073"),
    "geneval": ("https://github.com/djghosh13/geneval.git", "af4902f24d3ca90ebbb446dd9891a59e0f82725f"),
    "mmdetection_2x": ("https://github.com/open-mmlab/mmdetection.git", "e9cae2d0787cd5c2fc6165a6061f92fa09e48fb1"),
    "openai_clip": ("https://github.com/openai/CLIP.git", "d05afc436d78f1c48dc0dbf8e5980a9d471f35f6"),
}


def run(command: list[str], *, cwd: Path | None = None, env: dict[str, str] | None = None, capture: bool = False):
    print("+", " ".join(map(str, command)))
    return subprocess.run(
        [str(x) for x in command], cwd=str(cwd or REPO_ROOT), env=env,
        check=True, text=True, capture_output=capture,
    )


if RUN_INSTALLS:
    generation_packages = [
        "torch==2.10.0", "torchvision==0.25.0", "diffusers==0.30.2",
        "transformers==4.49.0", "accelerate==1.13.0", "huggingface-hub==0.29.3",
        "safetensors==0.7.0", "sentencepiece==0.2.1", "protobuf==5.29.3",
        "numpy==2.4.6", "Pillow==12.1.1", "pandas==3.0.0", "scipy==1.17.1",
        "scikit-learn==1.8.0", "matplotlib==3.10.9", "tqdm==4.67.1",
        "PyYAML==6.0.3", "rich==14.2.0", "einops==0.8.2", "datasets==4.5.0",
        "requests==2.32.5",
    ]
    run([sys.executable, "-m", "pip", "install", "--upgrade", "pip==24.3.1", "setuptools==75.6.0", "wheel==0.45.1"])
    run([sys.executable, "-m", "pip", "install", *generation_packages])
    run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(REPO_ROOT)])


+ /media/fezan/ASi/DVLM/steering/aim-flow/.venv/aim-flow/bin/python -m pip install --upgrade pip==24.3.1 setuptools==75.6.0 wheel==0.45.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 3.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 489.5 kB/s  0:00:01 eta 0:00:01
  Attempting uninstall: wheel
    Found existing installation: wheel 0.47.0
    Uninstalling wheel-0.47.0:
      Successfully uninstalled wheel-0.47.0
  Attempting uninstall: setuptools
    Found existing installation: setuptools 83.0.0
    Uninstalling setuptools-83.0.0:
      Successfully uninstalled setuptools-83.0.0━━━━━━━━━━━━━━━━━━ 1/3 [setuptools]
  Attempting uninstall: pip90m╺━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [setuptools]
    Found existing installation: pip 26.1.2━━━━━━━━━━━━━━━━━━━ 1/3 [setuptools]
    Uninstalling pip-26.1.2:━━━━━━╸━━━━━━━━━━━━━ 2/3 [pip]ls]
      Successfully uninstalled pip-26.1.2━━━━━━━━━━━━━ 2/3 [pip]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pi


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


+ /media/fezan/ASi/DVLM/steering/aim-flow/.venv/aim-flow/bin/python -m pip install --no-deps -e /media/fezan/ASi/DVLM/steering/aim-flow
Obtaining file:///media/fezan/ASi/DVLM/steering/aim-flow
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for aim-flow (pyproject.toml): started
  Building editable for aim-flow (pyproject.toml): finished with status 'done'
  Created wheel for aim-flow: filename=aim_flow-0.1.0-0.editable-py3-none-any.whl size=11533 sha256=534a1e43acc5532f585a2a45bf2031512a16f4d67da220044cf33ccac87fdbd1
  Stored


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
# Clone/verify all source repositories before any expensive work.
import json
import shutil


EXTERNAL = REPO_ROOT / "external"
REPO_PATHS = {
    "rectified_cfgpp": EXTERNAL / "Rectified-CFGpp",
    "t2i_compbench": EXTERNAL / "T2I-CompBench",
    "geneval": EXTERNAL / "GenEval",
    "mmdetection_2x": EXTERNAL / "mmdetection-2.x",
}


def git_output(repo: Path, *args: str) -> str:
    return subprocess.check_output(["git", "-C", str(repo), *args], text=True).strip()


def ensure_pinned_checkout(name: str, destination: Path) -> Path:
    url, commit = PINS[name]
    if not destination.exists():
        destination.parent.mkdir(parents=True, exist_ok=True)
        run(["git", "clone", "--filter=blob:none", url, str(destination)])
    if not (destination / ".git").exists():
        raise RuntimeError(f"{destination} exists but is not a git checkout")
    if git_output(destination, "status", "--porcelain", "--untracked-files=no"):
        raise RuntimeError(f"Refusing to change dirty official checkout: {destination}")
    if git_output(destination, "rev-parse", "HEAD") != commit:
        run(["git", "fetch", "origin", commit], cwd=destination)
        run(["git", "checkout", "--detach", commit], cwd=destination)
    actual = git_output(destination, "rev-parse", "HEAD")
    if actual != commit:
        raise RuntimeError(f"Pin mismatch for {name}: expected {commit}, found {actual}")
    return destination


for repo_name, repo_path in REPO_PATHS.items():
    ensure_pinned_checkout(repo_name, repo_path)

print(json.dumps({k: {"path": str(REPO_PATHS[k]), "commit": PINS[k][1]} for k in REPO_PATHS}, indent=2))


{
  "rectified_cfgpp": {
    "path": "/media/fezan/ASi/DVLM/steering/aim-flow/external/Rectified-CFGpp",
    "commit": "3c838882f7b2cdc2e1c3e5785468cdcb1fb27190"
  },
  "t2i_compbench": {
    "path": "/media/fezan/ASi/DVLM/steering/aim-flow/external/T2I-CompBench",
    "commit": "1b7094991a57f3c22abdd4f6e8ba6c1a15517073"
  },
  "geneval": {
    "path": "/media/fezan/ASi/DVLM/steering/aim-flow/external/GenEval",
    "commit": "af4902f24d3ca90ebbb446dd9891a59e0f82725f"
  },
  "mmdetection_2x": {
    "path": "/media/fezan/ASi/DVLM/steering/aim-flow/external/mmdetection-2.x",
    "commit": "e9cae2d0787cd5c2fc6165a6061f92fa09e48fb1"
  }
}


In [3]:
# Runtime imports, deterministic controls, hardware detection, and locked protocol.
import base64
import copy
import gc
import gzip
import hashlib
import importlib.metadata
import io
import math
import platform
import random
import re
import tempfile
import time
import urllib.request
from contextlib import redirect_stdout
from dataclasses import asdict, dataclass
from typing import Any, Iterable

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from huggingface_hub import HfFolder, model_info, snapshot_download

sys.path.insert(0, str(REPO_ROOT / "src"))

from aim_flow.eval_bench.generation import RectifiedCFGPPBackend, load_bench_config, unload_model
from aim_flow.eval_bench.constants import DEFAULT_SPFC_SCHEDULE_1_INDEXED
from aim_flow.prompt_schema import PrimitiveFlowSet
from aim_flow.sampler import AIMFlowSampler
from aim_flow.sd3_backend import SD3Backend


@dataclass(frozen=True)
class Protocol:
    model_id: str = "stabilityai/stable-diffusion-3-medium-diffusers"
    model_revision: str = "ea42f8cef0f178587cf766dc8129abd379c90671"
    scheduler: str = "checkpoint FlowMatchEulerDiscreteScheduler"
    dtype: str = "float16"
    load_t5_text_encoder: bool = False
    enable_model_cpu_offload: bool = True
    height: int = 1024
    width: int = 1024
    num_inference_steps: int = 28
    guidance_scale: float = 4.5
    negative_prompt: str = ""
    master_seed: int = 42
    t2i_samples_per_prompt: int = 10
    geneval_samples_per_prompt: int = 4
    rectified_sigma_noise: float = 0.005
    spfc_aggregation_steps_1_indexed: tuple[int, ...] = (1, 2, 3, 5, 6, 7, 8, 9, 10, 12, 14, 16, 19, 21, 23, 28)
    pipeline_batch_size: int = 1
    spfc_batch_size: int = 1


PROTOCOL = Protocol()
RECTIFIED_REFERENCE = {
    "official_project": "https://rectified-cfgpp.github.io/",
    "official_repository": PINS["rectified_cfgpp"][0],
    "paper": "https://arxiv.org/abs/2510.07631",
    "demo_model": "stabilityai/stable-diffusion-3.5-large",
    "paper_sd3_model": "stabilityai/stable-diffusion-3-medium",
    "width": 1024, "height": 1024, "num_inference_steps": 28,
    "guidance_scale": 4.5, "negative_prompt": "", "seed": 42,
    "sigma_noise": 0.005, "demo_batch_size": 1,
}
RECTIFIED_MODEL_ADAPTATION = (
    "Use the official Rectified-CFG++ pipeline with the paper-supported pinned SD3 Medium "
    "checkpoint so the backbone is identical across all three guidance methods."
)
METHODS = ("our_method", "base_cfg", "rectified_cfgpp")
METHOD_LABELS = {
    "our_method": "Our Method",
    "base_cfg": "Base CFG",
    "rectified_cfgpp": "Rectified CFG++",
}
T2I_CATEGORIES = ("color", "texture", "spatial", "shape")
T2I_SEEDS = tuple(PROTOCOL.master_seed + i for i in range(PROTOCOL.t2i_samples_per_prompt))
GENEVAL_SEEDS = tuple(PROTOCOL.master_seed + i for i in range(PROTOCOL.geneval_samples_per_prompt))

ARTIFACT_ROOT = Path(os.environ.get("AIM_FLOW_EVAL_ROOT", REPO_ROOT / "outputs" / "official_complete_evaluation")).resolve()
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
PRIMARY_GPU_INDEX = int(os.environ.get("AIM_FLOW_GPU", "0"))
OVERWRITE_INVALID = os.environ.get("AIM_FLOW_OVERWRITE_INVALID", "0") == "1"

HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN") or HfFolder.get_token()
if not HF_TOKEN:
    raise RuntimeError(
        "SD3 Medium is gated. Export HF_TOKEN (or log in with huggingface-cli) using an account "
        "that has accepted the model license. The token is never written to artifacts."
    )
try:
    checkpoint_info = model_info(PROTOCOL.model_id, revision=PROTOCOL.model_revision, token=HF_TOKEN)
    if checkpoint_info.sha != PROTOCOL.model_revision:
        raise RuntimeError(f"Resolved model revision {checkpoint_info.sha} != {PROTOCOL.model_revision}")
    MODEL_SNAPSHOT = Path(snapshot_download(
        PROTOCOL.model_id, revision=PROTOCOL.model_revision, token=HF_TOKEN,
        ignore_patterns=["text_encoder_3/*", "tokenizer_3/*"],
    )).resolve()
except Exception as exc:
    raise RuntimeError(f"Cannot access required checkpoint {PROTOCOL.model_id}: {exc}") from exc


def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(PROTOCOL.master_seed)
if torch.cuda.is_available():
    if PRIMARY_GPU_INDEX >= torch.cuda.device_count():
        raise ValueError(f"AIM_FLOW_GPU={PRIMARY_GPU_INDEX}, but only {torch.cuda.device_count()} CUDA devices exist")
    torch.cuda.set_device(PRIMARY_GPU_INDEX)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def sha256_file(path: Path, chunk_size: int = 1 << 20) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def canonical_hash(data: Any) -> str:
    return sha256_bytes(json.dumps(data, sort_keys=True, separators=(",", ":")).encode())


def remap_schedule(schedule: Iterable[int], source_steps: int, target_steps: int) -> tuple[int, ...]:
    if source_steps < 2 or target_steps < 2:
        raise ValueError("Schedule remapping requires at least two inference steps")
    mapped = tuple(round((step - 1) * (target_steps - 1) / (source_steps - 1)) + 1 for step in schedule)
    if len(mapped) != len(set(mapped)) or tuple(sorted(mapped)) != mapped:
        raise RuntimeError(f"Schedule remapping collapsed or reordered steps: {mapped}")
    return mapped


SPFC_BASE_SCHEDULE_1_INDEXED = tuple(DEFAULT_SPFC_SCHEDULE_1_INDEXED)
SPFC_SCHEDULE_MAPPING = {
    "source_num_inference_steps": 24,
    "source_schedule_1_indexed": SPFC_BASE_SCHEDULE_1_INDEXED,
    "target_num_inference_steps": PROTOCOL.num_inference_steps,
    "mapped_schedule_1_indexed": remap_schedule(
        SPFC_BASE_SCHEDULE_1_INDEXED, 24, PROTOCOL.num_inference_steps
    ),
    "mapping_rule": "round((step-1)*(target_steps-1)/(source_steps-1))+1",
}
if SPFC_SCHEDULE_MAPPING["mapped_schedule_1_indexed"] != PROTOCOL.spfc_aggregation_steps_1_indexed:
    raise RuntimeError("Locked SPFC schedule is not the declared normalized 24→28-step mapping")

RECTIFIED_SOURCE_FILES = {
    "demo.py": {"path": REPO_PATHS["rectified_cfgpp"] / "demo.py", "sha256": "6d9630c4cd9eb27a9c1958c5adb70af7a1b85acdaf84b37aba5c522075543146"},
    "pipeline.py": {"path": REPO_PATHS["rectified_cfgpp"] / "rect-cfg-SD3-pipeline" / "pipeline.py", "sha256": "fd83394f3a9aefbb2b9a706c785aab3dc5251d9b05a8022c5053f37704ef113a"},
}
for source_name, source in RECTIFIED_SOURCE_FILES.items():
    actual_hash = sha256_file(source["path"])
    if actual_hash != source["sha256"]:
        raise RuntimeError(f"Official Rectified-CFG++ {source_name} hash mismatch: {actual_hash}")

shared_rectified_defaults = {
    "width": PROTOCOL.width, "height": PROTOCOL.height,
    "num_inference_steps": PROTOCOL.num_inference_steps,
    "guidance_scale": PROTOCOL.guidance_scale,
    "negative_prompt": PROTOCOL.negative_prompt, "seed": PROTOCOL.master_seed,
    "sigma_noise": PROTOCOL.rectified_sigma_noise,
    "demo_batch_size": PROTOCOL.pipeline_batch_size,
}
for key, actual in shared_rectified_defaults.items():
    if actual != RECTIFIED_REFERENCE[key]:
        raise RuntimeError(f"Shared protocol diverges from official Rectified-CFG++ {key}: {actual}")


PROTOCOL_HASH = canonical_hash(asdict(PROTOCOL))
hardware = {
    "platform": platform.platform(), "python": sys.version, "cuda_available": torch.cuda.is_available(),
    "torch_cuda": torch.version.cuda, "cuda_device_count": torch.cuda.device_count(),
    "gpus": [
        {"index": i, "name": torch.cuda.get_device_name(i), "memory_bytes": torch.cuda.get_device_properties(i).total_memory}
        for i in range(torch.cuda.device_count())
    ],
}
if not torch.cuda.is_available():
    raise RuntimeError("SD3 Medium generation and both official benchmark scorers require CUDA.")
print(json.dumps({
    "protocol": asdict(PROTOCOL), "protocol_hash": PROTOCOL_HASH, "hardware": hardware,
    "rectified_cfgpp_reference": RECTIFIED_REFERENCE,
    "rectified_cfgpp_model_adaptation": RECTIFIED_MODEL_ADAPTATION,
    "spfc_schedule_mapping": SPFC_SCHEDULE_MAPPING,
}, indent=2))


/media/fezan/ASi/DVLM/steering/aim-flow/.venv/aim-flow/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 27 files: 100%|██████████| 27/27 [06:22<00:00, 14.15s/it]

{
  "protocol": {
    "model_id": "stabilityai/stable-diffusion-3-medium-diffusers",
    "model_revision": "ea42f8cef0f178587cf766dc8129abd379c90671",
    "scheduler": "checkpoint FlowMatchEulerDiscreteScheduler",
    "dtype": "float16",
    "load_t5_text_encoder": false,
    "enable_model_cpu_offload": true,
    "height": 1024,
    "width": 1024,
    "num_inference_steps": 28,
    "guidance_scale": 4.5,
    "negative_prompt": "",
    "master_seed": 42,
    "t2i_samples_per_prompt": 10,
    "geneval_samples_per_prompt": 4,
    "rectified_sigma_noise": 0.005,
    "spfc_aggregation_steps_1_indexed": [
      1,
      2,
      3,
      5,
      6,
      7,
      8,
      9,
      10,
      12,
      14,
      16,
      19,
      21,
      23,
      28
    ],
    "pipeline_batch_size": 1,
    "spfc_batch_size": 1
  },
  "protocol_hash": "5f2b71a5bb64a242164126cbe454b69a0401a4e8c23f58738053fbc8010c4d8e",
  "hardware": {
    "platform": "Linux-6.5.0-44-generic-x86_64-with-glibc2.38",
    "pyt

In [4]:
# Materialize the validated SPFC decompositions embedded in this notebook. This is data,
# not generated score output. It avoids requiring a local LLM during benchmark execution.
EMBEDDED_T2I_DECOMPOSITIONS_GZIP_B64 = "..."
EMBEDDED_GENEVAL_DECOMPOSITIONS_GZIP_B64 = "..."
EMBEDDED_T2I_DECOMPOSITIONS_SHA256 = "b3e8c109523a4e5231cfef8cc96ca41e9d32c55806a78818dc40be263b993cbb"
EMBEDDED_GENEVAL_DECOMPOSITIONS_SHA256 = "1e2bcedfa496f3214a22916eccc88b5cae03adbeec232a45e67045e4c40b2bb2"


def materialize_embedded_json(encoded: str, expected_sha256: str, destination: Path) -> dict[str, Any]:
    raw = gzip.decompress(base64.b64decode(encoded.encode("ascii")))
    actual = sha256_bytes(raw)
    if actual != expected_sha256:
        raise RuntimeError(f"Embedded decomposition checksum mismatch: {actual} != {expected_sha256}")
    data = json.loads(raw)
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(raw)
    return data


INPUT_ROOT = ARTIFACT_ROOT / "inputs"
T2I_DECOMPOSITIONS = materialize_embedded_json(
    EMBEDDED_T2I_DECOMPOSITIONS_GZIP_B64,
    EMBEDDED_T2I_DECOMPOSITIONS_SHA256,
    INPUT_ROOT / "t2i_selected_spfc_decompositions.json",
)
GENEVAL_DECOMPOSITIONS = materialize_embedded_json(
    EMBEDDED_GENEVAL_DECOMPOSITIONS_GZIP_B64,
    EMBEDDED_GENEVAL_DECOMPOSITIONS_SHA256,
    INPUT_ROOT / "geneval_spfc_decompositions.json",
)
print("T2I decompositions:", len(T2I_DECOMPOSITIONS["items"]))
print("GenEval decompositions:", len(GENEVAL_DECOMPOSITIONS["items"]))


T2I decompositions: 1200
GenEval decompositions: 553


In [5]:
# Load the official prompt files, preserve their exact order/metadata, and verify every
# SPFC target prompt against them before touching the model.
T2I_REPO = REPO_PATHS["t2i_compbench"]
GENEVAL_REPO = REPO_PATHS["geneval"]
T2I_PROMPT_HASHES = {
    "color": "1634259756dbc77d13093d907d414480080ec9790a8c97ad09efaea7b2534f2d",
    "texture": "fbb5363515b4a28009e360afaaaf389d2dd2289e50dbb0c6ef4ea0c7fe4f3d4f",
    "spatial": "8707a1d7e42ce3002d95363cf55f74bd043c843ae27108f5125aaaa7840ca988",
    "shape": "37e1a276906c7ea9516cbd7ac8501be896006c0c17ca7344f260562834455bae",
}
GENEVAL_METADATA_SHA256 = "5c48e0813e812e3c373fa5c8ed07a8f0a483be30272b4427b0559c8048e67c13"


def load_t2i_prompts() -> dict[str, list[str]]:
    result = {}
    for category in T2I_CATEGORIES:
        path = T2I_REPO / "examples" / "dataset" / f"{category}_val.txt"
        if sha256_file(path) != T2I_PROMPT_HASHES[category]:
            raise RuntimeError(f"Official prompt file changed despite repository pin: {path}")
        prompts = path.read_text(encoding="utf-8").splitlines()
        if len(prompts) != 300 or any(not p for p in prompts):
            raise RuntimeError(f"Expected 300 non-empty {category} prompts, found {len(prompts)}")
        if any("_" in p or "/" in p or "\\" in p for p in prompts):
            raise ValueError(f"{category} has a prompt incompatible with the official filename parser")
        result[category] = prompts
    return result


def load_geneval_metadata() -> list[dict[str, Any]]:
    path = GENEVAL_REPO / "prompts" / "evaluation_metadata.jsonl"
    if sha256_file(path) != GENEVAL_METADATA_SHA256:
        raise RuntimeError("Official GenEval metadata changed despite repository pin")
    records = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines()]
    if len(records) != 553 or any("prompt" not in x for x in records):
        raise RuntimeError(f"Expected 553 official GenEval records, found {len(records)}")
    return records


T2I_PROMPTS = load_t2i_prompts()
GENEVAL_METADATA = load_geneval_metadata()
GENEVAL_METADATA_LINES = (GENEVAL_REPO / "prompts" / "evaluation_metadata.jsonl").read_text(encoding="utf-8").splitlines()

t2i_decomp_by_key = {
    (item["metadata"]["category"], int(item["metadata"]["original_index"])): item
    for item in T2I_DECOMPOSITIONS["items"]
}
geneval_decomp_by_index = {int(item["metadata"]["official_index"]): item for item in GENEVAL_DECOMPOSITIONS["items"]}
if len(t2i_decomp_by_key) != 1200 or len(geneval_decomp_by_index) != 553:
    raise RuntimeError("Embedded decompositions are incomplete or have duplicate official indices")
for category, prompts in T2I_PROMPTS.items():
    for index, prompt in enumerate(prompts):
        if t2i_decomp_by_key[(category, index)]["target_prompt"] != prompt:
            raise RuntimeError(f"T2I decomposition prompt mismatch: {category}/{index}")
for index, metadata in enumerate(GENEVAL_METADATA):
    if geneval_decomp_by_index[index]["target_prompt"] != metadata["prompt"]:
        raise RuntimeError(f"GenEval decomposition prompt mismatch: {index}")
print("Official prompt/decomposition alignment: PASS")


Official prompt/decomposition alignment: PASS


In [6]:
# Common resumable generation interface. Valid PNG+sidecar pairs are never regenerated;
# corrupt or configuration-mismatched pairs cause a clear failure unless explicitly opted
# into replacement with AIM_FLOW_OVERWRITE_INVALID=1.
def write_json(data: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(data, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    os.replace(temporary, path)


def atomic_save_png(image: Image.Image, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp.png")
    image.save(temporary, format="PNG")
    with Image.open(temporary) as check:
        check.load()
        if check.size != (PROTOCOL.width, PROTOCOL.height):
            raise RuntimeError(f"Wrong image size from generator: {check.size}")
    os.replace(temporary, path)


def output_is_valid(image_path: Path, metadata_path: Path, expected: dict[str, Any]) -> bool:
    if not image_path.exists() and not metadata_path.exists():
        return False
    if image_path.exists() != metadata_path.exists():
        # A crash between the two atomic writes leaves a provably incomplete pair.
        # Remove only that partial artifact so interrupted runs resume automatically.
        image_path.unlink(missing_ok=True)
        metadata_path.unlink(missing_ok=True)
        return False
    valid = False
    try:
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        with Image.open(image_path) as image:
            image.verify()
        with Image.open(image_path) as image:
            valid = image.size == (PROTOCOL.width, PROTOCOL.height)
        valid = valid and all(metadata.get(k) == v for k, v in expected.items())
    except Exception:
        valid = False
    if not valid:
        if OVERWRITE_INVALID:
            image_path.unlink(missing_ok=True)
            metadata_path.unlink(missing_ok=True)
            return False
        raise RuntimeError(
            f"Incomplete or incompatible existing output: {image_path}. "
            "Inspect it or set AIM_FLOW_OVERWRITE_INVALID=1."
        )
    return True


def scheduler_record(pipe: Any) -> dict[str, Any]:
    scheduler = pipe.scheduler
    config = dict(scheduler.config)
    return {
        "class": f"{scheduler.__class__.__module__}.{scheduler.__class__.__name__}",
        "config": config,
        "fingerprint": canonical_hash(config),
    }


def make_config(seed: int):
    config = load_bench_config(
        seed=seed,
        aggregation_steps_1_indexed=list(PROTOCOL.spfc_aggregation_steps_1_indexed),
        num_inference_steps=PROTOCOL.num_inference_steps,
        height=PROTOCOL.height,
        width=PROTOCOL.width,
        guidance_scale=PROTOCOL.guidance_scale,
    )
    if config.model.model_id != PROTOCOL.model_id:
        raise RuntimeError("Repository benchmark model pin does not match notebook protocol")
    if tuple(step + 1 for step in config.primitive_flow.aggregation_steps) != PROTOCOL.spfc_aggregation_steps_1_indexed:
        raise RuntimeError("Repository config did not preserve the locked normalized SPFC schedule")
    locked = {
        "dtype": config.model.dtype == PROTOCOL.dtype,
        "load_t5_text_encoder": config.model.load_t5_text_encoder == PROTOCOL.load_t5_text_encoder,
        "enable_model_cpu_offload": config.model.enable_model_cpu_offload == PROTOCOL.enable_model_cpu_offload,
        "steps": config.sampler.num_inference_steps == PROTOCOL.num_inference_steps,
        "height": config.sampler.height == PROTOCOL.height, "width": config.sampler.width == PROTOCOL.width,
        "guidance_scale": config.sampler.guidance_scale == PROTOCOL.guidance_scale,
    }
    if not all(locked.values()):
        raise RuntimeError(f"Repository generation config violated locked protocol: {locked}")
    config.model.model_id = str(MODEL_SNAPSHOT)
    return config


class GuidanceRunner:
    def __init__(self, method: str):
        if method not in METHODS:
            raise ValueError(method)
        self.method = method
        self.config = make_config(PROTOCOL.master_seed)
        self.backend: Any = None
        self.sampler: AIMFlowSampler | None = None
        self.scheduler: dict[str, Any] | None = None

    def __enter__(self):
        seed_everything(PROTOCOL.master_seed)
        if self.method in {"our_method", "base_cfg"}:
            self.backend = SD3Backend(self.config).load()
            if self.method == "our_method":
                self.sampler = AIMFlowSampler(self.backend, self.config)
        else:
            self.backend = RectifiedCFGPPBackend(
                self.config,
                repo_dir=REPO_PATHS["rectified_cfgpp"],
                sigma_noise=PROTOCOL.rectified_sigma_noise,
            ).load()
        self.scheduler = scheduler_record(self.backend.pipe)
        if "FlowMatchEulerDiscreteScheduler" not in self.scheduler["class"]:
            raise RuntimeError(f"Unexpected scheduler: {self.scheduler['class']}")
        return self

    def __exit__(self, exc_type, exc, traceback):
        backend = self.backend
        self.sampler = None
        self.backend = None
        unload_model(backend)
        gc.collect()
        torch.cuda.empty_cache()

    def _base_or_rectified_batch(self, prompt: str, seeds: list[int]) -> list[Image.Image]:
        generators = [torch.Generator(device="cpu").manual_seed(seed) for seed in seeds]
        kwargs = {
            "prompt": prompt,
            "negative_prompt": PROTOCOL.negative_prompt,
            "height": PROTOCOL.height,
            "width": PROTOCOL.width,
            "num_inference_steps": PROTOCOL.num_inference_steps,
            "num_images_per_prompt": len(seeds),
            "generator": generators,
        }
        # Rectified CFG++ draws its intrinsic correction noise from the global CUDA
        # generator. Use a stable batch seed rather than Python's randomized hash().
        seed_everything(int(canonical_hash(seeds)[:8], 16))
        if self.method == "base_cfg":
            result = self.backend.pipe(guidance_scale=PROTOCOL.guidance_scale, **kwargs)
        else:
            result = self.backend.pipe(
                true_cfg=PROTOCOL.guidance_scale,
                sigma_noise=PROTOCOL.rectified_sigma_noise,
                **kwargs,
            )
        if len(result.images) != len(seeds):
            raise RuntimeError("Pipeline returned the wrong batch size")
        return result.images

    def generate_missing(
        self,
        prompt: str,
        decomposition: dict[str, Any] | None,
        outputs: list[tuple[int, Path, Path, dict[str, Any]]],
    ) -> None:
        missing = [entry for entry in outputs if not output_is_valid(entry[1], entry[2], entry[3])]
        if not missing:
            return
        if self.method == "our_method":
            if decomposition is None or self.sampler is None:
                raise RuntimeError("SPFC decomposition/sampler missing")
            for seed, image_path, metadata_path, expected in missing:
                self.config.sampler.seed = seed
                flow_set = PrimitiveFlowSet.from_dict({
                    "name": decomposition["id"],
                    "target_prompt": decomposition["target_prompt"],
                    "source_prompt": decomposition["source_prompt"],
                    "primitive_prompts": decomposition["primitive_prompts"],
                    "negative_prompt": PROTOCOL.negative_prompt,
                })
                started = time.perf_counter()
                image, method_metadata = self.sampler.generate_sparse_primitive_flow(flow_set, mode="primitive_flow_sparse")
                atomic_save_png(image, image_path)
                write_json({
                    **expected, "runtime_seconds": time.perf_counter() - started,
                    "scheduler": self.scheduler, "method_metadata": method_metadata,
                    "batch_size": PROTOCOL.spfc_batch_size,
                }, metadata_path)
        else:
            batch_size = PROTOCOL.pipeline_batch_size
            for start in range(0, len(missing), batch_size):
                chunk = missing[start:start + batch_size]
                seeds = [entry[0] for entry in chunk]
                started = time.perf_counter()
                images = self._base_or_rectified_batch(prompt, seeds)
                elapsed = time.perf_counter() - started
                for image, (seed, image_path, metadata_path, expected) in zip(images, chunk):
                    atomic_save_png(image, image_path)
                    write_json({
                        **expected, "runtime_seconds_batch": elapsed,
                        "scheduler": self.scheduler, "batch_size": len(chunk),
                        "rectified_cfgpp_commit": PINS["rectified_cfgpp"][1] if self.method == "rectified_cfgpp" else None,
                        "rectified_sigma_noise": PROTOCOL.rectified_sigma_noise if self.method == "rectified_cfgpp" else None,
                    }, metadata_path)


def expected_sidecar(benchmark: str, method: str, prompt: str, prompt_index: int, sample_index: int, seed: int) -> dict[str, Any]:
    return {
        "benchmark": benchmark, "method": method, "method_label": METHOD_LABELS[method],
        "prompt": prompt, "prompt_index": prompt_index, "sample_index": sample_index,
        "seed": seed, "protocol_hash": PROTOCOL_HASH, "negative_prompt": PROTOCOL.negative_prompt,
        "model_id": PROTOCOL.model_id, "model_revision": PROTOCOL.model_revision,
        "model_snapshot": str(MODEL_SNAPSHOT), "height": PROTOCOL.height, "width": PROTOCOL.width,
        "num_inference_steps": PROTOCOL.num_inference_steps, "guidance_scale": PROTOCOL.guidance_scale,
    }


## Part 1 — Official T2I-CompBench

Generation writes directly to the evaluator's required `samples/` structure. Filenames
are the exact prompt followed by the last-six-digit global question ID. The manifest and
per-image sidecars live beside (not inside) `samples/`, so the unmodified scorer sees
exactly 3,000 PNGs per method/category.


In [ ]:
def t2i_stage(method: str, category: str) -> Path:
    return ARTIFACT_ROOT / "t2i_compbench" / method / category


def run_t2i_generation() -> None:
    expected_total = len(METHODS) * len(T2I_CATEGORIES) * 300 * PROTOCOL.t2i_samples_per_prompt
    print(f"Expected T2I-CompBench images: {expected_total:,}")
    scheduler_fingerprints: dict[str, str] = {}
    for method in METHODS:
        with GuidanceRunner(method) as runner:
            scheduler_fingerprints[method] = runner.scheduler["fingerprint"]
            tasks = [(category, index, prompt) for category in T2I_CATEGORIES for index, prompt in enumerate(T2I_PROMPTS[category])]
            for category, prompt_index, prompt in tqdm(tasks, desc=f"T2I generation/{METHOD_LABELS[method]}", unit="prompt"):
                stage = t2i_stage(method, category)
                samples_dir = stage / "samples"
                metadata_dir = stage / "sample_metadata"
                outputs = []
                for sample_index, seed in enumerate(T2I_SEEDS):
                    question_id = prompt_index * PROTOCOL.t2i_samples_per_prompt + sample_index
                    image_path = samples_dir / f"{prompt}_{question_id:06d}.png"
                    metadata_path = metadata_dir / f"{question_id:06d}.json"
                    expected = expected_sidecar("t2i_compbench", method, prompt, prompt_index, sample_index, seed)
                    expected.update({"category": category, "question_id": question_id})
                    outputs.append((seed, image_path, metadata_path, expected))
                runner.generate_missing(prompt, t2i_decomp_by_key[(category, prompt_index)], outputs)
                manifest_path = stage / "prompt_manifest.jsonl"
                if not manifest_path.exists():
                    manifest_path.parent.mkdir(parents=True, exist_ok=True)
                    manifest_path.write_text(
                        "\n".join(json.dumps({"prompt_index": i, "prompt": p, "seeds": T2I_SEEDS}) for i, p in enumerate(T2I_PROMPTS[category])) + "\n",
                        encoding="utf-8",
                    )
    if len(set(scheduler_fingerprints.values())) != 1:
        raise RuntimeError(f"Scheduler config differs across methods: {scheduler_fingerprints}")
    write_json({
        "scheduler_fingerprints": scheduler_fingerprints, "protocol_hash": PROTOCOL_HASH,
        "rectified_cfgpp_reference": RECTIFIED_REFERENCE,
        "rectified_cfgpp_model_adaptation": RECTIFIED_MODEL_ADAPTATION,
        "rectified_cfgpp_source_hashes": {name: source["sha256"] for name, source in RECTIFIED_SOURCE_FILES.items()},
        "spfc_schedule_mapping": SPFC_SCHEDULE_MAPPING,
    }, ARTIFACT_ROOT / "t2i_compbench" / "generation_audit.json")


run_t2i_generation()


Expected T2I-CompBench images: 36,000


T2I generation/Our Method:   5%|▌         | 63/1200 [9:39:56<174:26:29, 552.32s/prompt]


KeyboardInterrupt: 

: 

In [ ]:
# Build an isolated Python 3.10 environment from the official T2I-CompBench requirements.
# The two git dependencies are commit-pinned without changing the official package list.
T2I_ENV = ARTIFACT_ROOT / ".envs" / "t2i_compbench_official"
T2I_ENV_PYTHON = T2I_ENV / "bin" / "python"
T2I_REQUIREMENTS_LOCK = ARTIFACT_ROOT / "environment_locks" / "t2i_compbench_requirements_pinned.txt"


def conda_executable() -> str:
    executable = shutil.which("conda")
    if not executable:
        raise FileNotFoundError("conda is required to create the pinned official evaluator environments")
    return executable


def capture_pip_freeze(python: Path, destination: Path) -> None:
    result = run([str(python), "-m", "pip", "freeze", "--all"], capture=True)
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(result.stdout, encoding="utf-8")


def ensure_t2i_environment() -> None:
    if not T2I_ENV_PYTHON.exists():
        run([conda_executable(), "create", "-y", "-p", str(T2I_ENV), "python=3.10", "pip=23.2.1"])
    official = (T2I_REPO / "requirements.txt").read_text(encoding="utf-8")
    official = official.replace(
        "git+https://github.com/openai/CLIP.git",
        f"git+https://github.com/openai/CLIP.git@{PINS['openai_clip'][1]}",
    )
    T2I_REQUIREMENTS_LOCK.parent.mkdir(parents=True, exist_ok=True)
    T2I_REQUIREMENTS_LOCK.write_text(official, encoding="utf-8")
    marker = T2I_ENV / ".aim_flow_requirements_complete"
    lock_hash = sha256_file(T2I_REQUIREMENTS_LOCK)
    if not marker.exists() or marker.read_text().strip() != lock_hash:
        run([str(T2I_ENV_PYTHON), "-m", "pip", "install", "-r", str(T2I_REQUIREMENTS_LOCK)])
        marker.write_text(lock_hash + "\n", encoding="utf-8")
    run([str(T2I_ENV_PYTHON), "-c", "import torch, transformers, detectron2, spacy; print(torch.__version__)"], capture=True)


ensure_t2i_environment()
capture_pip_freeze(T2I_ENV_PYTHON, ARTIFACT_ROOT / "environment_locks" / "t2i_compbench_pip_freeze.txt")


In [ ]:
# Download and verify the official UniDet RS200 checkpoint. BLIP-VQA downloads its
# checkpoint from the official URL pinned in the repository's vqa.yaml on first use.
UNIDET_WEIGHT = T2I_REPO / "UniDet_eval" / "experts" / "expert_weights" / "Unified_learned_OCIM_RS200_6x+2x.pth"
UNIDET_URL = "https://huggingface.co/shikunl/prismer/resolve/main/expert_weights/Unified_learned_OCIM_RS200_6x%2B2x.pth"
UNIDET_SHA256 = "b19a836811c0d0d7aebf04f83ff4ea1da77ccf57b85cd65819746c1fccbff258"


def download_verified(url: str, destination: Path, expected_sha256: str | None = None) -> Path:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if not destination.exists():
        temporary = destination.with_suffix(destination.suffix + ".part")
        print("Downloading", url)
        urllib.request.urlretrieve(url, temporary)
        os.replace(temporary, destination)
    actual = sha256_file(destination)
    if expected_sha256 and actual != expected_sha256:
        raise RuntimeError(f"Checkpoint checksum mismatch for {destination}: {actual}")
    print(destination, actual)
    return destination


download_verified(UNIDET_URL, UNIDET_WEIGHT, UNIDET_SHA256)


In [ ]:
# Execute the unmodified official scorers, save their logs/raw JSON separately, and
# aggregate the four category means. Scoring is resumable only when its provenance and
# raw result length match the exact generation protocol.
def evaluator_env() -> dict[str, str]:
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(PRIMARY_GPU_INDEX)
    env["PYTHONUNBUFFERED"] = "1"
    return env


def average_answers(path: Path, expected_count: int) -> float:
    data = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(data, list) or len(data) != expected_count:
        raise RuntimeError(f"Expected {expected_count} official scorer records in {path}, found {len(data) if isinstance(data, list) else type(data)}")
    values = [float(item["answer"]) for item in data]
    if not all(math.isfinite(x) for x in values):
        raise RuntimeError(f"Non-finite scorer output in {path}")
    return float(np.mean(values))


def run_logged(command: list[str], cwd: Path, log_path: Path, env: dict[str, str]) -> None:
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print("+", " ".join(command), f"(log: {log_path})")
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, cwd=str(cwd), env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        assert process.stdout is not None
        for line in process.stdout:
            log.write(line)
            if "score" in line.lower() or "processed" in line.lower():
                print(line.rstrip())
        return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)


def run_official_t2i_scoring() -> dict[str, dict[str, float]]:
    all_scores: dict[str, dict[str, float]] = {}
    expected_count = 300 * PROTOCOL.t2i_samples_per_prompt
    for method in METHODS:
        method_scores = {}
        for category in T2I_CATEGORIES:
            stage = t2i_stage(method, category)
            if len(list((stage / "samples").glob("*.png"))) != expected_count:
                raise RuntimeError(f"Generation incomplete: {method}/{category}")
            raw_dir = ARTIFACT_ROOT / "t2i_compbench" / method / category / "raw_scorer_outputs"
            provenance_path = raw_dir / "provenance.json"
            provenance = {
                "protocol_hash": PROTOCOL_HASH, "official_repo_commit": PINS["t2i_compbench"][1],
                "method": method, "category": category, "expected_count": expected_count,
            }
            if category == "spatial":
                command = [str(T2I_ENV_PYTHON), "2D_spatial_eval.py", "--outpath", str(stage)]
                cwd = T2I_REPO / "UniDet_eval"
                result_path = stage / "labels" / "annotation_obj_detection_2d" / "vqa_result.json"
            else:
                command = [str(T2I_ENV_PYTHON), "BLIP_vqa.py", "--out_dir", str(stage), "--np_num", "8"]
                cwd = T2I_REPO / "BLIPvqa_eval"
                result_path = stage / "annotation_blip" / "vqa_result.json"
            resume = False
            if provenance_path.exists() and result_path.exists():
                resume = json.loads(provenance_path.read_text()) == provenance
                if resume:
                    average_answers(result_path, expected_count)
            if not resume:
                run_logged(command, cwd, raw_dir / "official_scorer.log", evaluator_env())
                write_json(provenance, provenance_path)
            score = average_answers(result_path, expected_count)
            shutil.copy2(result_path, raw_dir / "vqa_result.json")
            method_scores[category] = score
            write_json({**provenance, "score": score, "raw_result": str(result_path)}, raw_dir / "score.json")
            gc.collect(); torch.cuda.empty_cache()
        method_scores["aggregate"] = float(np.mean([method_scores[c] for c in T2I_CATEGORIES]))
        all_scores[method] = method_scores
        write_json(method_scores, ARTIFACT_ROOT / "t2i_compbench" / method / "final_results.json")
    write_json({"protocol_hash": PROTOCOL_HASH, "scores": all_scores}, ARTIFACT_ROOT / "t2i_compbench" / "final_results.json")
    return all_scores


T2I_SCORES = run_official_t2i_scoring()
pd.DataFrame(T2I_SCORES).T


## Part 2 — Official GenEval generation and evaluation

The directory below is exactly the official GenEval layout: each prompt has a five-digit
folder, its original `metadata.jsonl`, and four five-digit sample PNGs. The official
Mask2Former evaluation and `summary_scores.py` consume these immutable PNGs directly.
No COCO image/caption set, FID, or auxiliary reward metric is used.


In [ ]:
def geneval_method_root(method: str) -> Path:
    return ARTIFACT_ROOT / "geneval" / method / "images"


def run_geneval_generation() -> None:
    expected_total = len(METHODS) * len(GENEVAL_METADATA) * PROTOCOL.geneval_samples_per_prompt
    print(f"Expected GenEval images: {expected_total:,}")
    scheduler_fingerprints: dict[str, str] = {}
    for method in METHODS:
        with GuidanceRunner(method) as runner:
            scheduler_fingerprints[method] = runner.scheduler["fingerprint"]
            for prompt_index, metadata in enumerate(tqdm(GENEVAL_METADATA, desc=f"GenEval generation/{METHOD_LABELS[method]}", unit="prompt")):
                prompt_dir = geneval_method_root(method) / f"{prompt_index:05d}"
                prompt_dir.mkdir(parents=True, exist_ok=True)
                metadata_path = prompt_dir / "metadata.jsonl"
                # Preserve the official metadata line byte-for-byte (plus its newline).
                exact_line = GENEVAL_METADATA_LINES[prompt_index] + "\n"
                if metadata_path.exists() and json.loads(metadata_path.read_text()) != metadata:
                    raise RuntimeError(f"Existing official metadata mismatch: {metadata_path}")
                metadata_path.write_text(exact_line, encoding="utf-8")
                outputs = []
                for sample_index, seed in enumerate(GENEVAL_SEEDS):
                    image_path = prompt_dir / "samples" / f"{sample_index:05d}.png"
                    sidecar_path = prompt_dir / "sample_metadata" / f"{sample_index:05d}.json"
                    expected = expected_sidecar("geneval", method, metadata["prompt"], prompt_index, sample_index, seed)
                    expected["official_metadata_sha256"] = sha256_bytes(exact_line.encode())
                    outputs.append((seed, image_path, sidecar_path, expected))
                runner.generate_missing(metadata["prompt"], geneval_decomp_by_index[prompt_index], outputs)
    if len(set(scheduler_fingerprints.values())) != 1:
        raise RuntimeError(f"Scheduler config differs across methods: {scheduler_fingerprints}")
    write_json({
        "scheduler_fingerprints": scheduler_fingerprints, "protocol_hash": PROTOCOL_HASH,
        "rectified_cfgpp_reference": RECTIFIED_REFERENCE,
        "rectified_cfgpp_model_adaptation": RECTIFIED_MODEL_ADAPTATION,
        "rectified_cfgpp_source_hashes": {name: source["sha256"] for name, source in RECTIFIED_SOURCE_FILES.items()},
        "spfc_schedule_mapping": SPFC_SCHEDULE_MAPPING,
    }, ARTIFACT_ROOT / "geneval" / "generation_audit.json")


run_geneval_generation()


In [ ]:
# Create the official GenEval conda environment, pin the official mmdetection 2.x checkout,
# and download the official Mask2Former detector.
GENEVAL_ENV = ARTIFACT_ROOT / ".envs" / "geneval_official"
GENEVAL_ENV_PYTHON = GENEVAL_ENV / "bin" / "python"
GENEVAL_MODEL_DIR = ARTIFACT_ROOT / "checkpoints" / "geneval_mask2former"
GENEVAL_MODEL = GENEVAL_MODEL_DIR / "mask2former_swin-s-p4-w7-224_lsj_8x2_50e_coco.pth"
GENEVAL_MODEL_URL = "https://download.openmmlab.com/mmdetection/v2.0/mask2former/mask2former_swin-s-p4-w7-224_lsj_8x2_50e_coco/mask2former_swin-s-p4-w7-224_lsj_8x2_50e_coco_20220504_001756-743b7d99.pth"


def ensure_geneval_environment() -> None:
    if not GENEVAL_ENV_PYTHON.exists():
        run([conda_executable(), "env", "create", "-y", "-p", str(GENEVAL_ENV), "-f", str(GENEVAL_REPO / "environment.yml")])
    marker = GENEVAL_ENV / ".mmdetection_2x_complete"
    if not marker.exists() or marker.read_text().strip() != PINS["mmdetection_2x"][1]:
        run([str(GENEVAL_ENV_PYTHON), "-m", "pip", "install", "-v", "-e", str(REPO_PATHS["mmdetection_2x"])])
        marker.write_text(PINS["mmdetection_2x"][1] + "\n", encoding="utf-8")
    run([str(GENEVAL_ENV_PYTHON), "-c", "import torch,mmdet,open_clip; print(torch.__version__,mmdet.__version__)"], capture=True)


ensure_geneval_environment()
capture_pip_freeze(GENEVAL_ENV_PYTHON, ARTIFACT_ROOT / "environment_locks" / "geneval_pip_freeze.txt")
download_verified(GENEVAL_MODEL_URL, GENEVAL_MODEL)
write_json({"url": GENEVAL_MODEL_URL, "sha256": sha256_file(GENEVAL_MODEL)}, GENEVAL_MODEL_DIR / "checkpoint.json")


In [ ]:
# Run the official GenEval evaluator and summary script. Raw per-image decisions, failure
# reasons, logs, task breakdowns, and overall scores remain separate for every method.
def parse_geneval_summary(text: str) -> dict[str, Any]:
    overall_match = re.search(r"Overall score \(avg\. over tasks\):\s*([0-9.]+)", text)
    image_match = re.search(r"% correct images:\s*([0-9.]+)%", text)
    prompt_match = re.search(r"% correct prompts:\s*([0-9.]+)%", text)
    if not overall_match:
        raise RuntimeError(f"Could not parse official GenEval summary:\n{text}")
    tasks = {name.strip(): float(value) / 100.0 for name, value in re.findall(r"^(.+?)\s*=\s*([0-9.]+)%", text, flags=re.MULTILINE)}
    return {
        "overall": float(overall_match.group(1)),
        "correct_images": float(image_match.group(1)) / 100.0 if image_match else None,
        "correct_prompts": float(prompt_match.group(1)) / 100.0 if prompt_match else None,
        "tasks": tasks,
        "stdout": text,
    }


def run_official_geneval_scoring() -> dict[str, dict[str, Any]]:
    summaries = {}
    expected = len(GENEVAL_METADATA) * PROTOCOL.geneval_samples_per_prompt
    for method in METHODS:
        output = ARTIFACT_ROOT / "geneval" / method / "official_evaluation"
        raw = output / "results.jsonl"
        provenance_path = output / "provenance.json"
        provenance = {
            "protocol_hash": PROTOCOL_HASH, "geneval_commit": PINS["geneval"][1],
            "mmdetection_commit": PINS["mmdetection_2x"][1], "detector_sha256": sha256_file(GENEVAL_MODEL),
            "expected_images": expected,
        }
        resume = False
        if raw.exists() and provenance_path.exists() and json.loads(provenance_path.read_text()) == provenance:
            resume = len(raw.read_text(encoding="utf-8").splitlines()) == expected
        if not resume:
            command = [
                str(GENEVAL_ENV_PYTHON), str(GENEVAL_REPO / "evaluation" / "evaluate_images.py"),
                str(geneval_method_root(method)), "--outfile", str(raw), "--model-path", str(GENEVAL_MODEL_DIR),
            ]
            run_logged(command, GENEVAL_REPO, output / "official_evaluator.log", evaluator_env())
            write_json(provenance, provenance_path)
        if len(raw.read_text(encoding="utf-8").splitlines()) != expected:
            raise RuntimeError(f"Official GenEval output incomplete: {raw}")
        result = run(
            [str(GENEVAL_ENV_PYTHON), str(GENEVAL_REPO / "evaluation" / "summary_scores.py"), str(raw)],
            cwd=GENEVAL_REPO, capture=True,
        )
        summary = parse_geneval_summary(result.stdout)
        write_json(summary, output / "summary.json")
        (output / "summary.txt").write_text(result.stdout, encoding="utf-8")
        summaries[method] = summary
        gc.collect(); torch.cuda.empty_cache()
    write_json(summaries, ARTIFACT_ROOT / "geneval" / "official_results.json")
    return summaries


GENEVAL_OFFICIAL = run_official_geneval_scoring()
pd.DataFrame({METHOD_LABELS[m]: {"Official GenEval": GENEVAL_OFFICIAL[m]["overall"]} for m in METHODS}).T


## Final official benchmark comparison and artifact export

This table is built only from completed outputs of the unmodified official T2I-CompBench and GenEval scorers. It cannot fabricate or fill missing values. Raw decisions, logs, and category/task summaries remain separated by method under the artifact root.


In [ ]:
GENEVAL_TASK_COLUMNS = {
    "single_object": "GenEval Single Object",
    "two_object": "GenEval Two Object",
    "counting": "GenEval Counting",
    "colors": "GenEval Colors",
    "position": "GenEval Position",
    "color_attr": "GenEval Color Attribution",
}

rows = []
for method in METHODS:
    official = GENEVAL_OFFICIAL[method]
    missing_tasks = sorted(set(GENEVAL_TASK_COLUMNS).difference(official["tasks"]))
    if missing_tasks:
        raise RuntimeError(f"Official GenEval summary is missing tasks for {method}: {missing_tasks}")
    row = {
        "Method": METHOD_LABELS[method],
        "T2I Color": T2I_SCORES[method]["color"],
        "T2I Texture": T2I_SCORES[method]["texture"],
        "T2I Spatial": T2I_SCORES[method]["spatial"],
        "T2I Shape": T2I_SCORES[method]["shape"],
        "T2I Aggregate": T2I_SCORES[method]["aggregate"],
        "GenEval Overall": official["overall"],
    }
    row.update({column: official["tasks"][task] for task, column in GENEVAL_TASK_COLUMNS.items()})
    rows.append(row)

FINAL_COLUMNS = [
    "Method", "T2I Color", "T2I Texture", "T2I Spatial", "T2I Shape", "T2I Aggregate",
    "GenEval Overall", *GENEVAL_TASK_COLUMNS.values(),
]
comparison = pd.DataFrame(rows, columns=FINAL_COLUMNS)
if comparison.drop(columns="Method").isna().any().any():
    raise RuntimeError("An official benchmark score is missing; refusing to export an incomplete table")
RESULTS_DIR = ARTIFACT_ROOT / "final_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
comparison.to_csv(RESULTS_DIR / "comparison.csv", index=False)
comparison.to_json(RESULTS_DIR / "comparison.json", orient="records", indent=2)
display(comparison)


In [ ]:
# Reproducibility manifest: source revisions, package versions, protocol, hardware,
# official inputs, metric/reference provenance, and exported artifact hashes.
important_packages = [
    "torch", "torchvision", "diffusers", "transformers", "accelerate", "huggingface-hub",
    "numpy", "pandas", "Pillow",
]
package_versions = {}
for package in important_packages:
    try:
        package_versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        package_versions[package] = None

manifest = {
    "protocol": asdict(PROTOCOL), "protocol_hash": PROTOCOL_HASH,
    "methods": METHOD_LABELS, "hardware": hardware, "package_versions": package_versions,
    "repository_pins": {k: {"url": url, "commit": commit} for k, (url, commit) in PINS.items()},
    "rectified_cfgpp_provenance": {
        "reference": RECTIFIED_REFERENCE,
        "model_adaptation": RECTIFIED_MODEL_ADAPTATION,
        "source_files": {
            name: {"path": str(source["path"]), "sha256": source["sha256"]}
            for name, source in RECTIFIED_SOURCE_FILES.items()
        },
    },
    "spfc_schedule_mapping": SPFC_SCHEDULE_MAPPING,
    "official_input_hashes": {"t2i": T2I_PROMPT_HASHES, "geneval": GENEVAL_METADATA_SHA256},
    "decomposition_hashes": {"t2i": EMBEDDED_T2I_DECOMPOSITIONS_SHA256, "geneval": EMBEDDED_GENEVAL_DECOMPOSITIONS_SHA256},
    "official_environment_locks": {
        "t2i_compbench": sha256_file(ARTIFACT_ROOT / "environment_locks" / "t2i_compbench_pip_freeze.txt"),
        "geneval": sha256_file(ARTIFACT_ROOT / "environment_locks" / "geneval_pip_freeze.txt"),
    },
    "official_geneval_scores": {m: {"overall": GENEVAL_OFFICIAL[m]["overall"], "tasks": GENEVAL_OFFICIAL[m]["tasks"]} for m in METHODS},
    "exports": {
        "csv": {"path": str(RESULTS_DIR / "comparison.csv"), "sha256": sha256_file(RESULTS_DIR / "comparison.csv")},
        "json": {"path": str(RESULTS_DIR / "comparison.json"), "sha256": sha256_file(RESULTS_DIR / "comparison.json")},
    },
}
write_json(manifest, RESULTS_DIR / "reproducibility_manifest.json")
print(json.dumps(manifest, indent=2))
